# Practical 06: Remembering Every Experiment

**Tracking runs, comparing them, and registering the winner with MLflow**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L11-L12 |
| **Course Outcome** | CO5 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 700 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. Record the settings and the score of every training run automatically.
2. Search past runs and identify the best one without opening a notebook.
3. Store a trained model as an artifact that can be loaded again later.
4. Register that model so it carries a name and a version number.

## What this practical is about

Six models were trained during P03. Stating now which settings produced the
best score is, for most people, not possible. The experiments were run, the
good number was held in memory, and the association between the two was lost.

**Experiment tracking** addresses exactly that loss. Each training run is
recorded as a row: its parameters, its metrics, the time it ran, and
optionally the trained object itself. The record is written by the code rather
than by the person, which is the property that makes it reliable.

Three terms are used throughout.

- A **run** is one execution of training, with its parameters and its metrics.
- An **experiment** is a named group of related runs.
- The **registry** is a separate list of named models, each with numbered
  versions, which is what later practicals load from.

The distinction between tracking and the registry matters. Tracking answers
what was tried. The registry answers what is to be used.

## Before you start

P02 to P04 should be complete. A model can be trained and scored. Familiarity
with a Python dictionary and a CSV file is assumed.

Run the cells from top to bottom. A later cell normally depends on an object
created by an earlier one, so a `NameError` usually indicates a skipped
prerequisite.

If notebook state becomes confusing, restart the kernel, clear all outputs, and
run again from the first cell.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'mlflow', 'matplotlib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'mlflow', 'matplotlib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

## Step 0b: ensure the shared dataset exists

Every practical in this course uses the same 600 synthetic delivery
observations. The next cell confirms the file is present and reconstructs it
from the course seed if it is absent.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
from pathlib import Path

def find_data(name="delivery_times.csv"):
    """Locate the shared dataset from any working directory.

    Walks upward from the current folder looking for data/<name>, so the
    notebook works whether it is launched from the practical folder, the
    Practicals folder, or the repository root."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder / "data" / name,
                          folder / "Practicals" / "data" / name):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{name} not found in any data/ folder at or above {here}")

DATA = find_data()
print("dataset:", DATA)


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

# Walkthrough

Read each step, then run its cell.

## Step 1: observe the problem

The cell below trains three models with different settings, as in P03. Read its
output, then attempt to recall the second model's settings without looking.
That recall is the quantity this practical replaces.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)

orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)

for n_trees, depth in [(50, 4), (150, 8), (300, 12)]:
    m = RandomForestRegressor(n_estimators=n_trees, max_depth=depth,
                              random_state=42).fit(X_train, y_train)
    mae = mean_absolute_error(y_test, m.predict(X_test))
    print(f"n_estimators={n_trees:3d} max_depth={depth:2d} -> "
          f"MAE {mae:.3f}")

Those values exist only in the cell output. Closing the notebook discards them.
The remaining steps record them instead.

## Step 2: point MLflow at a database

MLflow requires somewhere to write. This practical uses **SQLite**, a complete
database held in a single file and requiring no server process.

The choice of backend is not incidental. On MLflow 3.18 a filesystem path such
as `./mlruns` is refused outright, with the message that the filesystem
tracking backend is in maintenance mode and a database backend should be used
instead. Tutorials written before that change still show the filesystem path,
so the error is encountered often.

In [ ]:
import shutil

import mlflow
import mlflow.sklearn

# Teaching notebooks should start from a known state, so that your
# numbers match your instructor's. This clears any previous run of
# THIS notebook. You would never do this on a real project -- the
# whole value of the database is that it remembers.
db_path = (WORK / "mlflow.db").resolve()
db_path.unlink(missing_ok=True)
shutil.rmtree("mlartifacts", ignore_errors=True)

TRACKING_URI = f"sqlite:///{db_path.as_posix()}"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("delivery-time")

print("tracking store:", TRACKING_URI)

MLflow prints several `INFO` lines while it creates its database tables. That
output is expected on the first run.

An **experiment** is a named group of related runs. The one used here is called
`delivery-time`.

Note that re-running this cell removes the database and starts again. That is a
deliberate choice for teaching, so that every student begins from the same
state. Skip this cell to retain runs from a previous session.

## Step 3: record the first tracked run

`with mlflow.start_run():` opens a run. Everything logged inside the block is
attached to it, and the run is closed and persisted when the block ends.

Three kinds of value are logged. A **parameter** is a setting chosen before
training. A **metric** is a number measured after it. A **tag** is a label used
for grouping and filtering later.

In [ ]:
with mlflow.start_run(run_name="first-try") as run:
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("max_depth", 4)

    model = RandomForestRegressor(
        n_estimators=50, max_depth=4, random_state=42
    ).fit(X_train, y_train)

    mae = mean_absolute_error(y_test, model.predict(X_test))
    mlflow.log_metric("mae_minutes", mae)
    mlflow.set_tag("run_by", "student")

    first_run_id = run.info.run_id

print("run id :", first_run_id)
print("MAE    :", round(mae, 3))

That run is now held in the database permanently. Deleting this notebook leaves
the record intact, which is the difference between a record and an output.

## Step 4: read the runs back

`mlflow.search_runs` returns every run as a table. Parameters appear as
`params.<name>` columns and metrics as `metrics.<name>` columns, so the naming
in the query mirrors the naming in the logging call.

In [ ]:
runs = mlflow.search_runs(experiment_names=["delivery-time"])

print("runs recorded:", len(runs))
print()
print(runs[["run_id", "params.n_estimators", "params.max_depth",
            "metrics.mae_minutes"]].to_string(index=False))

That table was not saved by hand. It was assembled from the database on
request, which means it reflects the current state rather than the state at the
moment somebody chose to write it down.

## Step 5: log an entire sweep

The benefit becomes clear at scale. Wrapping the loop from Step 1 in tracking
records every combination as it is tried.

Note what is absent: no spreadsheet, no notes file, and no reliance on
recall.

In [ ]:
SETTINGS = [(50, 4), (150, 8), (300, 12), (100, 6), (200, None)]

for n_trees, depth in SETTINGS:
    with mlflow.start_run(run_name=f"rf-n{n_trees}-d{depth}"):
        mlflow.log_param("model_type", "RandomForestRegressor")
        mlflow.log_param("n_estimators", n_trees)
        mlflow.log_param("max_depth", depth)

        m = RandomForestRegressor(
            n_estimators=n_trees, max_depth=depth, random_state=42
        ).fit(X_train, y_train)

        preds = m.predict(X_test)
        mlflow.log_metric("mae_minutes",
                          mean_absolute_error(y_test, preds))
        mlflow.log_metric(
            "rmse_minutes",
            float(np.sqrt(((y_test - preds) ** 2).mean())),
        )
        print(f"  logged rf-n{n_trees}-d{depth}")

print("done")

## Step 6: identify the winner by querying

The question is put to the database directly: order by error ascending and take
the first row. This is the query a colleague would run months later, without
access to this notebook or to the person who ran it.

In [ ]:
best_runs = mlflow.search_runs(
    experiment_names=["delivery-time"],
    order_by=["metrics.mae_minutes ASC"],
)

columns = ["tags.mlflow.runName", "params.n_estimators",
           "params.max_depth", "metrics.mae_minutes"]
print(best_runs[columns].head(6).to_string(index=False))

winner = best_runs.iloc[0]
print()
print("WINNER:", winner["tags.mlflow.runName"],
      "with MAE", round(winner["metrics.mae_minutes"], 3))

Observe how close the leading three results are: approximately 2.38, 2.41 and
2.43 minutes. Tripling the number of trees purchased about one twentieth of a
minute.

MLflow did not establish that conclusion. It ensured the conclusion remains
checkable a month from now, when somebody asks whether the additional trees
justified the training time.

## Step 7: plot the sweep from the database

The MLflow web interface in Step 11 draws charts of this kind. The same chart
can be drawn directly from `search_runs`, and that is how a team's own
dashboard is in fact built: from the database, not from a spreadsheet and not
from memory.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

table = (best_runs[["tags.mlflow.runName", "metrics.mae_minutes"]]
         .sort_values("metrics.mae_minutes", ascending=False))
winner_name = winner["tags.mlflow.runName"]
colours = ["#2C5F79" if name == winner_name else "#6B7785"
           for name in table["tags.mlflow.runName"]]

fig, ax = plt.subplots(figsize=(8, 3.4))
bars = ax.barh(table["tags.mlflow.runName"],
               table["metrics.mae_minutes"], color=colours)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
ax.set_xlabel("MAE on the test set (minutes)")
ax.set_title("Every run MLflow remembers: worst at the top, winner in blue")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Six bars, one per run, read from the database. The lowest three are of nearly
equal length, which is the observation from Step 6 expressed as a picture. The
MLflow page opened in Step 11 presents the same comparison with less code.

## Step 8: store the winning model itself

Parameters and metrics describe a model. They are not the model. Using it later
requires storing the trained object as an **artifact**.

This cell takes longer than the others because MLflow also records the library
versions required to load the model again. Allow it half a minute.

Note the `skops_trusted_types` argument. MLflow 3 stores scikit-learn models
through skops, which refuses to load any class it has not been told to trust.
The declaration is made here, by the code that saves the model, rather than by
the code that loads it. That is the correct direction: the party who produced
the artifact is the party in a position to vouch for its contents.

In [ ]:
best_trees = int(winner["params.n_estimators"])
best_depth = winner["params.max_depth"]
best_depth = None if best_depth in ("None", None) else int(best_depth)

with mlflow.start_run(run_name="champion") as champion:
    mlflow.log_param("n_estimators", best_trees)
    mlflow.log_param("max_depth", best_depth)

    champion_model = RandomForestRegressor(
        n_estimators=best_trees, max_depth=best_depth,
        random_state=42,
    ).fit(X_train, y_train)

    mlflow.log_metric(
        "mae_minutes",
        mean_absolute_error(y_test, champion_model.predict(X_test)),
    )

    mlflow.sklearn.log_model(
        champion_model,
        name="model",
        input_example=X_train.head(3),
        registered_model_name="delivery-time-model",
        # Tree models store sklearn.tree._tree.Tree, which skops will not
        # load unless whoever saved it says it is trusted.
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )
    champion_id = champion.info.run_id

print("champion run:", champion_id)

`registered_model_name` performed two actions at once. It stored the model file
as an artifact of the run, and it entered that file into the **registry** under
a name.

## Step 9: inspect the registry

The registry is a separate list: named models, each carrying numbered versions.
Running this cell a second time produces version 2, which is the behaviour the
registry exists to provide. A version is never overwritten.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

for mv in client.search_model_versions(
    "name='delivery-time-model'"
):
    print(f"name={mv.name}  version={mv.version}  "
          f"run_id={mv.run_id[:8]}...")

## Step 10: load a model back by name and version

This is the result the preceding steps exist to deliver. Loading the model
requires neither the notebook that trained it, nor the variable it occupied, nor
the person who ran it. It requires a name and a version.

P07 applies this same mechanism to place a model behind a web address.

In [ ]:
loaded = mlflow.sklearn.load_model("models:/delivery-time-model/1")

one_order = pd.DataFrame([{
    "distance_km": 5.0,
    "prep_time_min": 20,
    "traffic_level": 2,
    "rain": 0,
}])

print("loaded model type:", type(loaded).__name__)
print("prediction:", round(float(loaded.predict(one_order)[0]), 1),
      "minutes")

## Step 11: view the runs in the web interface

Everything logged has a web page. The server is not started inside the
notebook, because a web server would occupy the cell indefinitely.

Open a terminal, change to this practical's folder, and run the command printed
below. Then visit <http://127.0.0.1:5000> in a browser. Press `Ctrl + C` in the
terminal to stop the server when finished.

In [ ]:
print("Run this in a terminal, from the folder containing work/ :")
print()
print(f"  cd {Path.cwd()}")
print("  python -m mlflow ui --backend-store-uri "
      f"{TRACKING_URI} --port 5000")
print()
print("Then open http://127.0.0.1:5000")
print()
print("What you will see first: an empty 'Experiments' page.")
print("Click 'delivery-time' in the left sidebar to get the run")
print("table. Tick two runs and press Compare to see them side")
print("by side. The Models tab at the top shows the registry.")

Take a screenshot of the run table with the `mae_minutes` column visible. It is
required for the submission.

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It may be run
as often as required and it never alters the notebook.

## Task T1: track a different kind of model

Log one new run, named exactly `linear-baseline`, which

- trains a `LinearRegression` on `X_train` and `y_train`,
- logs a parameter `model_type` with the value `LinearRegression`,
- logs a metric `mae_minutes` holding its error on the test set.

Store the run's identifier in `T1_run_id`.

> **Hint.** Follow Step 3 and change three things: the run name, the model, and
> the value of `model_type`. `LinearRegression` is imported from
> `sklearn.linear_model`. The identifier is available as `run.info.run_id`
> inside the `with` block.

In [ ]:
from sklearn.linear_model import LinearRegression

T1_run_id = None

# TODO: open a run called "linear-baseline", train a LinearRegression,
#       log model_type and mae_minutes, and save the run id.

print("T1_run_id =", T1_run_id)

## Task T2: query the database rather than the output

Without consulting any earlier output, use `mlflow.search_runs` to find the run
with the lowest `mae_minutes` across the whole `delivery-time` experiment.

Place its run name in `T2_best_name` and its error in `T2_best_mae`.

> **Hint.** Step 6 shows the pattern. Pass
> `order_by=["metrics.mae_minutes ASC"]` and then take `.iloc[0]`. The run name
> is held in the column `tags.mlflow.runName`.

In [ ]:
# TODO: query the runs sorted by mae_minutes, smallest first
T2_runs = None

# TODO: the name and the error of the best one
T2_best_name = None
T2_best_mae = None

print(T2_best_name, T2_best_mae)

## Task T3: a sweep that can be filtered afterwards

Run a sweep of `DecisionTreeRegressor` at depths `[2, 4, 6]`. For each depth,
log one run which

- carries the tag `family` set to `tree`,
- logs the parameter `max_depth`,
- logs the metric `mae_minutes`.

Then query MLflow for runs carrying that tag and place the count in
`T3_tree_runs`.

> **Hint.** `mlflow.set_tag("family", "tree")` is called inside the `with`
> block. To retrieve them, pass `filter_string="tags.family = 'tree'"` to
> `search_runs`. The count is the length of the result.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# TODO: log one tagged run per depth in [2, 4, 6]

# TODO: count the runs tagged family=tree
T3_tree_runs = None

print("runs tagged family=tree:", T3_tree_runs)

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | a run named linear-baseline exists in MLflow', lambda: len(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.mlflow.runName = 'linear-baseline'")) >= 1)
_check("T1 | T1_run_id is that run's id", lambda: T1_run_id in set(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.mlflow.runName = 'linear-baseline'")['run_id']))
_check('T1 | it logged model_type and a sensible mae_minutes', lambda: (lambda r: r['params.model_type'].iloc[0] == 'LinearRegression' and 0 < float(r['metrics.mae_minutes'].iloc[0]) < 15)(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.mlflow.runName = 'linear-baseline'")))
_check('T2 | T2_best_mae really is the smallest error recorded', lambda: abs(float(T2_best_mae) - float(mlflow.search_runs(experiment_names=['delivery-time'])['metrics.mae_minutes'].min())) < 1e-6)
_check('T2 | T2_best_name is the run that holds it', lambda: T2_best_name == mlflow.search_runs(experiment_names=['delivery-time'], order_by=['metrics.mae_minutes ASC']).iloc[0]['tags.mlflow.runName'])
_check('T3 | three runs are tagged family=tree', lambda: len(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.family = 'tree'")) == 3)
_check('T3 | T3_tree_runs reports that count', lambda: int(T3_tree_runs) == 3)
_check('T3 | each tagged run logged a max_depth of 2, 4 or 6', lambda: set(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.family = 'tree'")['params.max_depth']) == {'2', '4', '6'})

print("==================================================================")
print("SELF-CHECK   Practical 06: Remembering Every Experiment")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. A screenshot of the MLflow run table in the browser, showing the
   `mae_minutes` column and at least six runs.
3. In a markdown cell, one sentence stating which settings won and by what
   margin they beat the worst run.

Name the file `P06_<your-roll-number>.ipynb` before uploading it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, runs visible in MLflow | 3 |
| Task T1, a linear regression run logged correctly | 2 |
| Task T2, the best run found by querying rather than by eye | 2 |
| Task T3, a tagged sweep that can be filtered | 3 |
| **Total** | **10** |

## Read more

- MLflow, tracking quickstart: <https://mlflow.org/docs/latest/getting-started/intro-quickstart/>
- MLflow, search runs syntax: <https://mlflow.org/docs/latest/search-runs.html>
- MLflow, model registry: <https://mlflow.org/docs/latest/model-registry.html>
- MLflow, backend stores: <https://mlflow.org/docs/latest/self-hosting/>